# Briscola Benchmark Analysis
Runs the Rust benchmark binary and loads results directly into pandas — no temp files.

In [ ]:
import io
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

WORKSPACE = Path("__file__").resolve().parent.parent.parent  # cardroom/
BINARY = WORKSPACE / "target" / "release" / "examples" / "benchmark"

sns.set_theme(style="darkgrid")
plt.rcParams["figure.dpi"] = 120

In [ ]:
def build():
    """Build the release binary. Only needed once per code change."""
    subprocess.run(
        ["cargo", "build", "--release", "--example", "benchmark"],
        cwd=WORKSPACE, check=True,
    )
    print("Build OK:", BINARY)


def run_benchmark(agent0: str, agent1: str, n_rounds: int = 1000) -> pd.DataFrame:
    """Run the benchmark and return per-trick data as a DataFrame.

    Progress is printed to stderr (visible in the notebook).
    CSV data is captured from stdout.
    """
    result = subprocess.run(
        [str(BINARY), agent0, agent1, str(n_rounds), "--csv"],
        capture_output=True, text=True, cwd=WORKSPACE, check=True,
    )
    df = pd.read_csv(io.StringIO(result.stdout))
    df["had_briscola"] = df["had_briscola"].astype(bool)
    df["agent0_took_trick"] = df["agent0_took_trick"].astype(bool)
    return df

In [ ]:
# Build once; comment out after first run if you're only changing the notebook
build()

In [ ]:
# --- Configure your matchup here ---
AGENT0 = "bot"
AGENT1 = "random"
N_ROUNDS = 2000

df = run_benchmark(AGENT0, AGENT1, N_ROUNDS)
print(f"{len(df):,} trick rows  ({N_ROUNDS * 2:,} games)")
df.head()

## Win rates & average final points

In [ ]:
# One row per game (use last trick of each game)
games = df.groupby(["round", "game"]).last().reset_index()

n_games = len(games)
wins0 = (games["game_winner"] == "agent0").sum()
wins1 = (games["game_winner"] == "agent1").sum()
draws = (games["game_winner"] == "draw").sum()

print(f"Games: {n_games}")
print(f"{AGENT0:>10}  wins: {wins0:5}  ({wins0/n_games*100:.1f}%)")
print(f"{AGENT1:>10}  wins: {wins1:5}  ({wins1/n_games*100:.1f}%)")
print(f"{'Draw':>10}       : {draws:5}  ({draws/n_games*100:.1f}%)")
print()
print(f"Avg final pts  {AGENT0}: {games['agent0_final'].mean():.1f}")
print(f"Avg final pts  {AGENT1}: {games['agent1_final'].mean():.1f}")

## Points progression across tricks

In [ ]:
progression = df.groupby("trick")[["agent0_cumpts", "agent1_cumpts"]].mean()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(progression.index, progression["agent0_cumpts"], label=AGENT0, linewidth=2)
ax.plot(progression.index, progression["agent1_cumpts"], label=AGENT1, linewidth=2)
ax.axhline(60, color="gray", linestyle="--", linewidth=1, label="50% (60 pts)")
ax.set_xlabel("Trick number")
ax.set_ylabel("Avg cumulative points")
ax.set_title(f"Points progression — {AGENT0} vs {AGENT1} ({n_games:,} games)")
ax.legend()
ax.set_xticks(range(1, 21))
plt.tight_layout()
plt.show()

## Points gained per trick

In [ ]:
gained = df.copy()
# Compute per-trick gain from cumulative columns
gained = gained.sort_values(["round", "game", "trick"])
gained["agent0_gained"] = gained.groupby(["round", "game"])["agent0_cumpts"].diff().fillna(gained["agent0_cumpts"])
gained["agent1_gained"] = gained.groupby(["round", "game"])["agent1_cumpts"].diff().fillna(gained["agent1_cumpts"])

avg_gained = gained.groupby("trick")[["agent0_gained", "agent1_gained"]].mean()

fig, ax = plt.subplots(figsize=(10, 5))
x = avg_gained.index
width = 0.4
ax.bar(x - width/2, avg_gained["agent0_gained"], width, label=AGENT0)
ax.bar(x + width/2, avg_gained["agent1_gained"], width, label=AGENT1)
ax.set_xlabel("Trick number")
ax.set_ylabel("Avg points gained")
ax.set_title(f"Avg points gained per trick — {AGENT0} vs {AGENT1}")
ax.legend()
ax.set_xticks(range(1, 21))
plt.tight_layout()
plt.show()

## Briscola tricks: points at stake vs non-briscola

In [ ]:
briscola_stats = df.groupby("had_briscola")["trick_points"].agg(["mean", "median", "std", "count"])
briscola_stats.index = ["no briscola", "had briscola"]
print(briscola_stats.round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Avg trick points by briscola presence
briscola_stats["mean"].plot.bar(ax=axes[0], color=["steelblue", "darkorange"], rot=0)
axes[0].set_ylabel("Avg points at stake")
axes[0].set_title("Avg trick points: briscola vs none")
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f"{bar.get_height():.1f}", ha="center", va="bottom", fontsize=11)

# Distribution of trick points by briscola presence
df[df["had_briscola"]]["trick_points"].plot.hist(
    ax=axes[1], bins=30, alpha=0.6, label="had briscola", color="darkorange")
df[~df["had_briscola"]]["trick_points"].plot.hist(
    ax=axes[1], bins=30, alpha=0.6, label="no briscola", color="steelblue")
axes[1].set_xlabel("Points at stake")
axes[1].set_title("Distribution of trick points")
axes[1].legend()

plt.suptitle(f"{AGENT0} vs {AGENT1} — {n_games:,} games", y=1.02)
plt.tight_layout()
plt.show()

## Win rate on briscola tricks per agent

In [ ]:
trick_wins = df.groupby(["trick", "had_briscola"])["agent0_took_trick"].mean().reset_index()
trick_wins["agent1_took_trick"] = 1 - trick_wins["agent0_took_trick"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, briscola, title in zip(
    axes,
    [False, True],
    ["Non-briscola tricks", "Briscola tricks"],
):
    sub = trick_wins[trick_wins["had_briscola"] == briscola]
    ax.plot(sub["trick"], sub["agent0_took_trick"], label=AGENT0, linewidth=2)
    ax.plot(sub["trick"], sub["agent1_took_trick"], label=AGENT1, linewidth=2)
    ax.axhline(0.5, color="gray", linestyle="--", linewidth=1)
    ax.set_xlabel("Trick number")
    ax.set_ylabel("Win rate")
    ax.set_title(title)
    ax.legend()
    ax.set_xticks(range(1, 21))

plt.suptitle(f"Trick win rate by trick number — {AGENT0} vs {AGENT1}", y=1.02)
plt.tight_layout()
plt.show()

## Final score distributions

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
bins = range(0, 125, 5)
ax.hist(games["agent0_final"], bins=bins, alpha=0.6, label=AGENT0)
ax.hist(games["agent1_final"], bins=bins, alpha=0.6, label=AGENT1)
ax.axvline(60, color="gray", linestyle="--", linewidth=1, label="60 pts")
ax.set_xlabel("Final points")
ax.set_ylabel("Games")
ax.set_title(f"Final score distribution — {AGENT0} vs {AGENT1} ({n_games:,} games)")
ax.legend()
plt.tight_layout()
plt.show()

## Points gained specifically on briscola tricks

In [ ]:
briscola_tricks = gained[gained["had_briscola"]]

avg_briscola = briscola_tricks.groupby("trick")[["agent0_gained", "agent1_gained"]].mean()

fig, ax = plt.subplots(figsize=(10, 5))
x = avg_briscola.index
ax.bar(x - 0.2, avg_briscola["agent0_gained"], 0.4, label=AGENT0)
ax.bar(x + 0.2, avg_briscola["agent1_gained"], 0.4, label=AGENT1)
ax.set_xlabel("Trick number")
ax.set_ylabel("Avg points gained")
ax.set_title(f"Avg points gained on briscola tricks — {AGENT0} vs {AGENT1}")
ax.legend()
plt.tight_layout()
plt.show()